<a href="https://colab.research.google.com/github/mmatinm/SmartCane-AI-Visual-Navigation-System/blob/main/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Functions

In [5]:
import os
import yaml
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

def report_yolo_dataset(dataset_dir, show_examples=True, num_examples=8):
    def count_images(split):
        img_dir = os.path.join(dataset_dir, split, "images")
        if os.path.exists(img_dir):
            return len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        else:
            return 0

    def analyze_split_labels(split, num_classes):
        label_dir = os.path.join(dataset_dir, split, "labels")
        class_image_counts = [0] * num_classes
        overlap_count = 0
        empty_or_missing_count = 0

        image_dir = os.path.join(dataset_dir, split, "images")
        if not os.path.exists(image_dir):
            return class_image_counts, overlap_count, empty_or_missing_count

        image_files = [f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

        for img_file in image_files:
            img_stem = os.path.splitext(img_file)[0]
            label_path = os.path.join(label_dir, img_stem + ".txt")
            classes_in_image = set()

            if os.path.exists(label_path):
                with open(label_path, 'r') as f:
                    lines = f.readlines()
                    for line in lines:
                        parts = line.strip().split()
                        if len(parts) >= 1:
                            cls_id = int(float(parts[0]))
                            if 0 <= cls_id < num_classes:
                                classes_in_image.add(cls_id)
            else:
                empty_or_missing_count += 1
                continue  # no label file

            if len(classes_in_image) == 0:
                empty_or_missing_count += 1
            else:
                if len(classes_in_image) > 1:
                    overlap_count += 1
                for cls_id in classes_in_image:
                    class_image_counts[cls_id] += 1

        return class_image_counts, overlap_count, empty_or_missing_count

    # Print image counts
    train_count = count_images("train")
    valid_count = count_images("valid")
    test_count = count_images("test")

    print(f"📊 Image counts:")
    print(f"  Train: {train_count}")
    print(f"  Validation: {valid_count}")
    print(f"  Test: {test_count}")

    # Load class names
    data_yaml_path = os.path.join(dataset_dir, "data.yaml")
    with open(data_yaml_path, 'r') as f:
        data_yaml = yaml.safe_load(f)

    class_names = data_yaml.get("names", [])
    num_classes = len(class_names)

    print("\n🏷️ Classes in dataset:")
    for i, cls in enumerate(class_names):
        print(f"  {i}: {cls}")

    # Count per-class image occurrences and extra stats
    print("\n📂 Per-class image counts (number of images where each class appears):")
    for split in ["train", "valid", "test"]:
        class_image_counts, overlap_count, empty_count = analyze_split_labels(split, num_classes)
        print(f"  {split.capitalize()}:")
        for i, count in enumerate(class_image_counts):
            print(f"    {class_names[i]} ({i}): {count}")
        print(f"    ➕ Overlap images (contain ≥2 classes): {overlap_count}")
        print(f"    🚫 Images without any class (empty or missing labels): {empty_count}")

    if not show_examples:
        return

    # Pick examples from train split
    image_dir = os.path.join(dataset_dir, "train", "images")
    label_dir = os.path.join(dataset_dir, "train", "labels")

    image_files = [f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if len(image_files) == 0:
        print("\n⚠️ No images found in train/images to display.")
        return

    sample_files = random.sample(image_files, min(num_examples, len(image_files)))

    print(f"\n🖼️ Showing {len(sample_files)} random annotated images...")

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()

    for ax, fname in zip(axes, sample_files):
        img_path = os.path.join(image_dir, fname)
        label_path = os.path.join(label_dir, os.path.splitext(fname)[0] + ".txt")

        image = Image.open(img_path)
        w, h = image.size
        ax.imshow(image)
        ax.axis('off')

        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cls_id, xc, yc, bw, bh = map(float, parts)
                    x = (xc - bw / 2) * w
                    y = (yc - bh / 2) * h
                    box_w = bw * w
                    box_h = bh * h
                    rect = patches.Rectangle((x, y), box_w, box_h,
                                             linewidth=2, edgecolor='red', facecolor='none')
                    ax.add_patch(rect)
                    if class_names:
                        ax.text(x, y, class_names[int(cls_id)], color='yellow', fontsize=10,
                                bbox=dict(facecolor='black', alpha=0.5))

    plt.tight_layout()
    plt.show()


# Merging

In [1]:

!cp -r "/content/drive/MyDrive/SmartCane/final_bus" /content/
!cp -r "/content/drive/MyDrive/SmartCane/final_crosswalk" /content/
!cp -r "/content/drive/MyDrive/SmartCane/final_escalator" /content/
!cp -r "/content/drive/MyDrive/SmartCane/final_rglight" /content/
!cp -r "/content/drive/MyDrive/SmartCane/final_safety" /content/


In [7]:
import os
import shutil
import yaml
from tqdm import tqdm

# List of dataset paths
datasets = [
    "/content/final_bus",
    "/content/final_crosswalk",
    "/content/final_escalator",
    "/content/final_rglight",
    "/content/final_safety"
]

# Output folder
output_dir = "/content/merged1"
splits = ["train", "valid", "test"]

# Make merged folders
for split in splits:
    os.makedirs(os.path.join(output_dir, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, split, "labels"), exist_ok=True)

global_class_names = []
class_offset = 0

# Merge process
for dataset in datasets:
    dataset_name = os.path.basename(dataset)

    # Read class names from data.yaml
    data_yaml_path = os.path.join(dataset, "data.yaml")
    with open(data_yaml_path, "r") as f:
        data_yaml = yaml.safe_load(f)

    local_names = data_yaml.get("names", [])
    if isinstance(local_names, dict):
        # convert dict to list
        local_names = [local_names[k] for k in sorted(local_names.keys())]

    # Append local names to global list
    global_class_names.extend(local_names)

    for split in splits:
        img_dir = os.path.join(dataset, split, "images")
        lbl_dir = os.path.join(dataset, split, "labels")
        out_img_dir = os.path.join(output_dir, split, "images")
        out_lbl_dir = os.path.join(output_dir, split, "labels")

        if not os.path.exists(img_dir):
            continue

        image_files = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        pbar = tqdm(image_files, desc=f"Merging {split} from {dataset_name}", ncols=100)

        for img_file in pbar:
            base_name = os.path.splitext(img_file)[0]
            src_img_path = os.path.join(img_dir, img_file)
            src_lbl_path = os.path.join(lbl_dir, base_name + ".txt")

            # Rename with dataset prefix to avoid name clashes
            new_base = f"{dataset_name}_{base_name}"
            dst_img_path = os.path.join(out_img_dir, new_base + ".jpg")
            dst_lbl_path = os.path.join(out_lbl_dir, new_base + ".txt")

            shutil.copy2(src_img_path, dst_img_path)

            if os.path.exists(src_lbl_path):
                with open(src_lbl_path, "r") as f:
                    lines = f.readlines()

                new_lines = []
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    old_cls = int(parts[0])
                    new_cls = old_cls + class_offset
                    new_line = f"{new_cls} " + " ".join(parts[1:]) + "\n"
                    new_lines.append(new_line)

                with open(dst_lbl_path, "w") as f:
                    f.writelines(new_lines)

    class_offset += len(local_names)

# Write final merged data.yaml
final_yaml = {
    "train": os.path.join(output_dir, "train/images"),
    "val": os.path.join(output_dir, "valid/images"),
    "test": os.path.join(output_dir, "test/images"),
    "nc": len(global_class_names),
    "names": global_class_names
}

with open(os.path.join(output_dir, "data.yaml"), "w") as f:
    yaml.dump(final_yaml, f)

print("✅ Merge complete with class remapping.")


Merging test from final_safety: 100%|█████████████████████████████| 82/82 [00:00<00:00, 1439.70it/s]

✅ Merge complete with class remapping.


In [ ]:
report_yolo_dataset('/content/drive/MyDrive/SmartCane/final_dataset')

In [9]:
import shutil

# Source and destination paths
src = "/content/merged1"
dst = "/content/drive/MyDrive/SmartCane/final_dataset"

# Copy folder recursively
shutil.copytree(src, dst)
print(f"✅ Dataset copied to {dst}")

✅ Dataset copied to /content/drive/MyDrive/SmartCane/final_dataset


# Preprocessing

In [15]:

!cp -r "/content/drive/MyDrive/SmartCane/final_dataset" /content/

In [52]:
import os
import cv2
import yaml
import shutil
import random
from glob import glob
from tqdm import tqdm
import albumentations as A

# Define augmentation pipeline
def get_augmentations():
    return A.Compose([
        A.Resize(640, 640),
        A.RandomBrightnessContrast(p=0.3),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=15, p=0.4),
        A.Blur(p=0.2),
        A.CLAHE(p=0.1),
    ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.3))

# Load YOLO-format label
def load_yolo_label(label_path):
    boxes = []
    with open(label_path, 'r') as f:
        for line in f.readlines():
            parts = line.strip().split()
            if len(parts) >= 5:
                cls, x, y, w, h = map(float, parts[:5])
                boxes.append([x, y, w, h, int(cls)])
    return boxes

# Save YOLO-format label
def save_yolo_label(label_path, boxes):
    with open(label_path, 'w') as f:
        for box in boxes:
            x, y, w, h, cls = box
            f.write(f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")

# Augment and save
def augment_and_save(img_path, label_path, out_img_dir, out_label_dir, num_aug=3):
    image = cv2.imread(img_path)
    h, w = image.shape[:2]
    boxes = load_yolo_label(label_path)

    for i in range(num_aug):
        bboxes = [[x, y, bw, bh] for x, y, bw, bh, _ in boxes]
        class_labels = [cls for *_, cls in boxes]

        aug = get_augmentations()
        try:
            augmented = aug(image=image, bboxes=bboxes, class_labels=class_labels)
        except:
            continue

        new_img = augmented['image']
        new_boxes = augmented['bboxes']
        new_labels = augmented['class_labels']

        if not new_boxes:
            continue

        out_img_name = os.path.splitext(os.path.basename(img_path))[0] + f"_{i}.jpg"
        out_label_name = os.path.splitext(os.path.basename(label_path))[0] + f"_{i}.txt"

        cv2.imwrite(os.path.join(out_img_dir, out_img_name), new_img)
        save_yolo_label(os.path.join(out_label_dir, out_label_name),
                        [[*box, cls] for box, cls in zip(new_boxes, new_labels)])

# Preprocess dataset
def preprocess_yolo_dataset(input_dir, output_dir):
    splits = ['train', 'valid', 'test']
    for split in splits:
        print(f"Processing split: {split}")
        image_dir = os.path.join(input_dir, split, 'images')
        label_dir = os.path.join(input_dir, split, 'labels')

        out_img_dir = os.path.join(output_dir, split, 'images')
        out_label_dir = os.path.join(output_dir, split, 'labels')
        os.makedirs(out_img_dir, exist_ok=True)
        os.makedirs(out_label_dir, exist_ok=True)

        for img_path in tqdm(glob(os.path.join(image_dir, '*.jpg'))):
            base_name = os.path.splitext(os.path.basename(img_path))[0]
            label_path = os.path.join(label_dir, base_name + '.txt')

            if not os.path.exists(label_path):
                continue

            if split == 'train':
                # Apply 3 augmentations to each training image
                augment_and_save(img_path, label_path, out_img_dir, out_label_dir, num_aug=3)
            else:
                # For valid/test: just resize the image once without augmentation
                augment_and_save(img_path, label_path, out_img_dir, out_label_dir, num_aug=1)

# Generate data.yaml
def create_data_yaml(output_dir, class_names):
    data = {
        'train': os.path.join(output_dir, 'train', 'images'),
        'val': os.path.join(output_dir, 'valid', 'images'),
        'test': os.path.join(output_dir, 'test', 'images'),
        'nc': len(class_names),
        'names': class_names
    }

    with open(os.path.join(output_dir, 'data.yaml'), 'w') as f:
        yaml.dump(data, f)

def extract_class_names_from_yaml(yaml_path):
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    return data['names']

# Example usage

input_dataset = '/content/final_dataset'
output_dataset = '/content/augmented_dataset400'

# Dynamically extract class names
yaml_path = '/content/final_dataset/data.yaml'
class_names = extract_class_names_from_yaml(yaml_path)


preprocess_yolo_dataset(input_dataset, output_dataset)
create_data_yaml(output_dataset, class_names)


Processing split: train


100%|██████████| 7676/7676 [06:06<00:00, 20.96it/s]


Processing split: valid


100%|██████████| 1236/1236 [00:24<00:00, 49.86it/s]


Processing split: test


100%|██████████| 1482/1482 [00:28<00:00, 52.53it/s]


In [54]:
import shutil

# Source and destination paths
src = "/content/augmented_dataset400"
dst = "/content/drive/MyDrive/SmartCane/final_dataset_augmented"

# Copy folder recursively
shutil.copytree(src, dst)
print(f"✅ Dataset copied to {dst}")

✅ Dataset copied to /content/drive/MyDrive/SmartCane/final_dataset_augmented


In [ ]:
report_yolo_dataset('/content/drive/MyDrive/SmartCane/final_dataset_augmented')

In [46]:
import shutil

#shutil.rmtree('/content/augmented_dataset200')


# Train

In [ ]:
!pip install ultralytics


In [ ]:
from ultralytics import YOLO

# Load model (you can change to yolov8m.pt or yolov8l.pt)
model = YOLO('yolov8n.pt')


In [51]:

# Train
model.train(
    data='/content/augmented_dataset400/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,  # 0 means first GPU
    name='yolov8_custom',
    project='/content/yolo_train_output',
    workers=4
)



Ultralytics 8.3.175 🚀 Python-3.11.13 torch-2.6.0+cu124 CPU (Intel Xeon 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/augmented_dataset200/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8_custom3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose

RuntimeError: Dataset '/content/augmented_dataset200/data.yaml' error ❌ /content/augmented_dataset200/data.yaml 'val:' key missing ❌.
'train' and 'val' are required in all data YAMLs.

# Test

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
import numpy as np
import torch

def evaluate_yolov8(model_path, data_yaml, split='test'):
    from ultralytics import YOLO

    model = YOLO(model_path)
    results = model.val(data=data_yaml, split=split, save_json=True, save_txt=True, iou=0.5)

    # Access metrics
    metrics = results.metrics
    names = results.names

    print("\n📊 Classification Report")
    print(f"Precision: {metrics.precision:.3f}")
    print(f"Recall: {metrics.recall:.3f}")
    print(f"mAP@0.5: {metrics.map50:.3f}")
    print(f"mAP@0.5:0.95: {metrics.map:.3f}")

    # Plot confusion matrix
    matrix = metrics.confusion_matrix.matrix.astype(int)
    disp = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=names.values())
    disp.plot(cmap=plt.cm.Blues, xticks_rotation=45)
    plt.title("Confusion Matrix")
    plt.show()


In [ ]:
evaluate_yolov8(
    model_path='/content/yolo_train_output/yolov8_custom/weights/best.pt',
    data_yaml='/content/merged1/datasets/data.yaml'
)
